In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Funkcja Qiskit firmy Qedma
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium, Flex oraz On-Prem (za pośrednictwem IBM Quantum Platform API). Mają status wydania podglądowego i mogą ulec zmianie.

## Przegląd
Choć procesory kwantowe (QPU) znacznie się poprawiły w ostatnich latach, błędy wynikające z szumów i niedoskonałości istniejącego sprzętu nadal stanowią kluczowe wyzwanie dla twórców algorytmów kwantowych. W miarę jak dziedzina zbliża się do kwantowych obliczeń w skali użytkowej, których nie można zweryfikować klasycznie, rozwiązania umożliwiające eliminację szumów z gwarantowaną dokładnością stają się coraz ważniejsze. Aby sprostać temu wyzwaniu, Qedma opracowała metodę Quantum Error Mitigation (QESEM), bezproblemowo zintegrowaną z IBM Quantum Platform jako [Funkcja Qiskit](/guides/functions).

Dzięki QESEM możesz uruchamiać swoje obwody kwantowe na zaszumionych QPU i uzyskiwać wysoce dokładne wyniki wolne od błędów przy minimalnym narzucie czasowym QPU, bliskim fundamentalnym granicom. Aby to osiągnąć, QESEM korzysta z zestawu autorskich metod opracowanych przez Qedmę, służących do charakteryzacji i redukcji błędów. Techniki redukcji błędów obejmują optymalizację Gate, transpilację uwzględniającą szumy, tłumienie błędów (ES) oraz nieobciążone łagodzenie błędów (EM). Dzięki połączeniu tych metod opartych na charakteryzacji możesz uzyskiwać wiarygodne, wolne od błędów wyniki dla ogólnych kwantowych Circuit o dużej objętości, otwierając zastosowania niemożliwe do realizacji w inny sposób.

Pełny opis składowych oraz demonstrację w skali użytkowej znajdziesz w artykule [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Opis
Możesz używać funkcji QESEM firmy Qedma, aby łatwo szacować i wykonywać swoje Circuit z tłumieniem i łagodzeniem błędów, osiągając większe objętości Circuit i wyższą dokładność. Aby korzystać z QESEM, podajesz Circuit kwantowy, zestaw obserwowalnych do zmierzenia, docelową dokładność statystyczną dla każdej obserwowalnej oraz wybrany QPU. Zanim uruchomisz Circuit do docelowej dokładności, możesz oszacować wymagany czas QPU na podstawie obliczeń analitycznych, które nie wymagają wykonania Circuit. Gdy jesteś zadowolony z oszacowania czasu QPU, możesz wykonać Circuit za pomocą QESEM.

Gdy wykonujesz Circuit, QESEM uruchamia protokół charakteryzacji urządzenia dostosowany do twojego Circuit, uzyskując wiarygodny model szumu dla błędów występujących w Circuit. Na podstawie charakteryzacji QESEM najpierw implementuje transpilację uwzględniającą szumy, aby odwzorować wejściowy Circuit na zestaw fizycznych Qubit i Gate, minimalizując szum wpływający na docelową obserwowalną. Obejmuje to natywnie dostępne Gate (CX/CZ na urządzeniach IBM&reg;) oraz dodatkowe Gate zoptymalizowane przez QESEM, tworzące rozszerzony zestaw Gate QESEM. Następnie QESEM uruchamia zestaw Circuit ES i EM opartych na charakteryzacji na QPU i zbiera wyniki pomiarów. Są one następnie przetwarzane klasycznie, aby dostarczyć nieobciążoną wartość oczekiwaną i słupek błędu dla każdej obserwowalnej, odpowiadające żądanej dokładności.

![Przegląd Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Wykazano, że QESEM zapewnia wyniki o wysokiej dokładności dla różnorodnych zastosowań kwantowych i na największych objętościach Circuit osiągalnych dziś. QESEM oferuje następujące funkcje dostępne dla użytkownika, zademonstrowane w sekcji benchmarków poniżej:
-	**Gwarantowana dokładność:** QESEM generuje nieobciążone szacowania wartości oczekiwanych obserwowalnych. Jego metoda EM jest wyposażona w teoretyczne gwarancje, które – w połączeniu z najnowocześniejszą charakteryzacją Qedmy – zapewniają, że łagodzenie błędów zbiega do wyjścia Circuit bez szumów z dokładnością określoną przez użytkownika. W przeciwieństwie do wielu heurystycznych metod EM podatnych na systematyczne błędy lub odchylenia, gwarantowana dokładność QESEM jest niezbędna do uzyskiwania wiarygodnych wyników dla ogólnych Circuit kwantowych i obserwowalnych.
-	**Skalowalność do dużych QPU:** Czas QPU QESEM zależy od objętości Circuit, ale poza tym jest niezależny od liczby Qubit. Qedma zademonstrowała QESEM na największych dostępnych dziś urządzeniach kwantowych, w tym IBM Quantum Eagle o 127 Qubit i Heron o 133 Qubit.
-	**Agnostyczność względem zastosowań:** QESEM zademonstrowano na różnorodnych zastosowaniach, w tym symulacji Hamiltonianu, VQE, QAOA i szacowaniu amplitudy. Możesz wprowadzić dowolny Circuit kwantowy i obserwowalną do zmierzenia i uzyskać dokładne wyniki wolne od błędów. Jedyne ograniczenia wynikają ze specyfikacji sprzętu i przydzielonego czasu QPU, które określają dostępne objętości Circuit i dokładności wyjścia. W przeciwieństwie do tego wiele rozwiązań redukcji błędów jest specyficznych dla danego zastosowania lub obejmuje niekontrolowane heurystyki, co sprawia, że nie nadają się do ogólnych Circuit kwantowych i zastosowań.
-  **Rozszerzony zestaw Gate:** QESEM obsługuje Gate z ułamkowymi kątami i udostępnia zoptymalizowane przez Qedmę Gate $Rzz(\theta)$ z ułamkowymi kątami na urządzeniach IBM Quantum Heron i Eagle. Ten rozszerzony zestaw Gate umożliwia efektywniejszą kompilację i odblokowuje objętości Circuit większe nawet o czynnik 2 w porównaniu z domyślną kompilacją CX/CZ.
-	**Wielobazowe obserwowalne:** QESEM obsługuje obserwowalne wejściowe złożone z wielu niekomutujących ciągów Pauliego, takich jak ogólne Hamiltoniany. Wybór baz pomiarowych oraz optymalizacja alokacji zasobów QPU (strzały i Circuit) są następnie wykonywane automatycznie przez QESEM w celu minimalizacji wymaganego czasu QPU dla żądanej dokładności. Ta optymalizacja, uwzględniająca wierności sprzętu i szybkości wykonania, umożliwia uruchamianie głębszych Circuit i uzyskiwanie wyższych dokładności.
## Benchmarki
QESEM zostało przetestowane na szerokiej gamie przypadków użycia i zastosowań. Poniższe przykłady mogą pomóc ci ocenić, jakie typy obciążeń możesz uruchamiać z QESEM.

Kluczową miarą ilościową trudności zarówno łagodzenia błędów, jak i klasycznej symulacji dla danego Circuit i obserwowalnej jest **aktywna objętość**: liczba bramek CNOT wpływających na obserwowalną w Circuit. Aktywna objętość zależy od głębokości i szerokości Circuit, od wagi obserwowalnej oraz od struktury Circuit, która określa stożek świetlny obserwowalnej. Więcej szczegółów znajdziesz w prezentacji z [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM zapewnia szczególnie dużą wartość w reżimie wysokiej objętości, dając wiarygodne wyniki dla ogólnych Circuit i obserwowalnych.

![Aktywna objętość](../docs/images/guides/qedma-qesem/active_volume.svg)

| Zastosowanie | Liczba Qubit | Urządzenie | Opis Circuit | Dokładność | Czas całkowity | Użycie Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuit VQE  | 8              | Eagle (r3) | 21 warstw łącznie, 9 baz pomiarowych, łańcuch 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 unikalne warstwy x 3 kroki, topologia 2D heavy-hex                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 unikalne warstwy x 8 kroków, topologia 2D heavy-hex                     | 97%      | 116 min      | 23 min          |
| Symulacja Hamiltonianu metodą Trottera   | 40  | Eagle (r3)            | 2 unikalne warstwy x 10 kroków Trottera, łańcuch 1D                    | 97%      | 3 godziny     | 25 min         |
| Symulacja Hamiltonianu metodą Trottera   | 119  | Eagle (r3)           | 3 unikalne warstwy x 9 kroków Trottera, topologia 2D heavy-hex                    | 95%      | 6,5 godziny     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 unikalne warstwy x 15 kroków, topologia 2D heavy-hex                    | 99%      | 52 min             | 9 min           |

Dokładność mierzona jest tutaj względem idealnej wartości obserwowalnej: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, gdzie '$\epsilon$' to bezwzględna precyzja łagodzenia błędów (ustawiana przez użytkownika), a $\langle O \rangle_{ideal}$ to obserwowalna dla Circuit bez szumów.
„Użycie Runtime" mierzy zużycie benchmarku w trybie wsadowym (suma użycia poszczególnych zadań), natomiast „czas całkowity" mierzy użycie w trybie Session (czas ścienny eksperymentu), który obejmuje dodatkowe czasy klasyczne i komunikacyjne. QESEM jest dostępny do wykonania w obu trybach, dzięki czemu możesz jak najlepiej wykorzystać dostępne zasoby.

Obwody Kicked Ising o 28 Qubit symulują Dyskretny Kwaziryształ Czasowy badany przez Shinjo et al. (zob. [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) i [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) na trzech połączonych pętlach ibm_kawasaki. Parametry Circuit przyjęte tutaj to $(\theta_x, \theta_z) = (0.9 \pi, 0)$, ze ferromagnetycznym stanem początkowym $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Mierzoną obserwowalną jest wartość bezwzględna magnetyzacji $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Eksperyment Kicked Ising w skali użytkowej przeprowadzono na 136 najlepszych Qubit urządzenia ibm_fez; ten konkretny benchmark przeprowadzono przy kącie Clifforda $(\theta_x, \theta_z) = (\pi, 0)$, przy którym aktywna objętość rośnie wolno wraz z głębokością Circuit, co – w połączeniu z wysoką wiernością urządzenia – umożliwia wysoką dokładność przy krótkim czasie działania.

Obwody symulacji Hamiltonianu metodą Trottera dotyczą modelu Isinga z polem poprzecznym przy ułamkowych kątach: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ i $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ odpowiednio (zob. [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Circuit w skali użytkowej uruchomiono na 119 najlepszych Qubit urządzenia ibm_brisbane, natomiast eksperyment z 40 Qubit przeprowadzono na najlepszym dostępnym łańcuchu. Dokładność jest podana dla magnetyzacji; wyniki o wysokiej dokładności uzyskano również dla obserwowalnych o większej wadze.

Circuit VQE został opracowany wspólnie z badaczami z Centrum Technologii Kwantowych i Zastosowań w Deutsches Elektronen-Synchrotron (DESY). Docelową obserwowalną był tutaj Hamiltonian składający się z dużej liczby niekomutujących ciągów Pauliego, co podkreśla zoptymalizowaną wydajność QESEM dla obserwowalnych wielobazowych. Łagodzenie błędów zastosowano do klasycznie zoptymalizowanego ansatzu; choć wyniki te są jeszcze nieopublikowane, wyniki tej samej jakości zostaną uzyskane dla różnych Circuit o podobnych właściwościach strukturalnych.
## Pierwsze kroki
Uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz Funkcję Qiskit QESEM w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w swoim lokalnym środowisku.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Przykłady
Aby zacząć, wypróbuj ten podstawowy przykład szacowania wymaganego czasu QPU do uruchomienia QESEM dla danego `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Poniższy przykład wykonuje zadanie QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Możesz używać znanych interfejsów API Qiskit Serverless, aby sprawdzić status obciążenia swojej Funkcji Qiskit lub zwrócić wyniki:

In [ ]:
print(sample_job.status())
result = sample_job.result()

Poniższy fragment kodu pokazuje, jak pobrać szacowany czas QPU (gdy `estimate_time_only` jest ustawione):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Poniższy fragment kodu pokazuje, jak pobrać wyniki mitigacji (gdy `estimate_time_only` nie jest ustawione) oraz metryki wykonania. Zawierają one kluczowe dane umożliwiające głębsze zrozumienie wpływu różnych parametrów na wykonanie QESEM. Mogą być również przydatne podczas pisania artykułu naukowego opartego na twoich badaniach.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Pobieranie komunikatów o błędach
Jeśli status twojego zadania to ERROR, użyj `job.result()`, aby pobrać komunikat o błędzie:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).


</Admonition>